# 🧪 Laboratorio 9: Análisis Temporal de Indicadores Económicos — Soluciones

**Módulo 03** | **Sesión 6** | **Duración estimada: 45min**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-03-modelado-predictivo/laboratorios/lab_09_series_temporales_soluciones.ipynb)

---

Este notebook contiene las **soluciones completas** con código, interpretaciones y comentarios didácticos.

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Descripción | Nivel |
|---|-------------|-------------|-------|
| 1 | Series temporales | Cargar, preparar y visualizar datos temporales | Aplicación |
| 2 | Suavizamiento | Calcular y visualizar medias móviles | Aplicación |
| 3 | Descomposición | Descomponer una serie en tendencia, estacionalidad y residuo | Análisis |
| 4 | Estacionariedad | Evaluar estacionariedad con la prueba ADF | Análisis |
| 5 | Pronóstico | Ajustar un modelo ARIMA y generar pronósticos | Aplicación |

## 💡 ¿Por qué es importante para tu práctica docente?

El **análisis de series temporales** es una competencia esencial para economistas y administradores. En Venezuela, comprender la dinámica de indicadores como la inflación, el tipo de cambio y la liquidez monetaria es fundamental tanto para la investigación académica como para la toma de decisiones.

En este laboratorio trabajarás con **datos del Banco Central de Venezuela** (BCV) para:

- Visualizar la evolución temporal de indicadores clave
- Identificar tendencias y patrones estacionales
- Evaluar si una serie es estacionaria
- Generar pronósticos a corto plazo

Estos son exactamente los análisis que se realizan en materias como Macroeconomía, Econometría y Política Económica.

In [ ]:
# Librerías necesarias — ejecuta esta celda primero
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('Librerías cargadas correctamente ✓')
print('\nNota: si statsmodels no está instalado, ejecuta:')
print('  !pip install statsmodels')

---

## Ejercicio 1: Cargar datos y visualizar la inflación

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 1: Cargar datos y visualizar la inflación
# ============================================================

# 1. Cargar el CSV
df = pd.read_csv('../../datasets/economia/indicadores_bcv.csv')
print(f'📊 Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Columnas: {list(df.columns)}')
display(df.head())

# 2. Convertir la columna fecha a tipo datetime
df['fecha'] = pd.to_datetime(df['fecha'])

# 3. Establecer fecha como índice
df = df.set_index('fecha')

# 4. Ordenar cronológicamente
df = df.sort_index()

print(f'\nRango temporal: {df.index.min().strftime("%Y-%m")} a {df.index.max().strftime("%Y-%m")}')
print(f'Total de meses: {len(df)}')

# 5. Gráfico de línea de inflacion_mensual
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df.index, df['inflacion_mensual'], color='steelblue', linewidth=1.5)

# 6. Título, etiquetas y grid
ax.set_xlabel('Fecha')
ax.set_ylabel('Inflación mensual (%)')
ax.set_title('Evolución de la Inflación Mensual — Venezuela (BCV)')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📌 La serie muestra la dinámica inflacionaria a lo largo del tiempo.')
print(f'   Inflación promedio: {df["inflacion_mensual"].mean():.2f}%')
print(f'   Inflación máxima:   {df["inflacion_mensual"].max():.2f}%')
print(f'   Inflación mínima:   {df["inflacion_mensual"].min():.2f}%')

---

## Ejercicio 2: Media móvil de 6 meses

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 2: Media móvil de 6 meses
# ============================================================

# 1. Calcular la media móvil de 6 meses
#    rolling(window=6) toma los últimos 6 valores y calcula su media
#    Los primeros 5 valores serán NaN (no hay suficientes datos previos)
media_movil_6 = df['inflacion_mensual'].rolling(window=6).mean()

# 2. Crear el gráfico con ambas series
fig, ax = plt.subplots(figsize=(12, 6))

# Serie original (semitransparente para que se vea la tendencia)
ax.plot(df.index, df['inflacion_mensual'],
        color='steelblue', alpha=0.4, linewidth=1,
        label='Inflación mensual (original)')

# Media móvil (línea gruesa roja)
ax.plot(df.index, media_movil_6,
        color='red', linewidth=2.5,
        label='Media móvil 6 meses')

# 3. Leyenda y título
ax.set_xlabel('Fecha')
ax.set_ylabel('Inflación mensual (%)')
ax.set_title('Inflación Mensual con Media Móvil de 6 Meses — Venezuela')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. Comentario sobre la tendencia
print('📌 Observaciones:')
print('  - La media móvil suaviza las fluctuaciones mensuales y revela la tendencia.')
print('  - La línea roja muestra la dirección general de la inflación.')
print('  - Los picos en la serie original se atenúan con el suavizamiento.')

---

## Ejercicio 3: Descomposición de la serie

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 3: Descomposición de la serie
# ============================================================

# Preparar la serie: eliminar NaN para que la descomposición funcione
serie_inflacion = df['inflacion_mensual'].dropna()

# 1. Descomposición aditiva con periodo = 12 (datos mensuales)
#    Modelo aditivo: Y(t) = Tendencia + Estacionalidad + Residuo
descomposicion = seasonal_decompose(
    serie_inflacion,
    model='additive',
    period=12
)

# 2. Visualizar los 4 componentes
fig = descomposicion.plot()
fig.set_size_inches(12, 10)
fig.suptitle('Descomposición de la Serie de Inflación', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# 3. Comentario interpretativo
print('📌 Interpretación de la descomposición:')
print('  - Observado: la serie original de inflación mensual.')
print('  - Tendencia: la dirección general a largo plazo de la inflación.')
print('  - Estacionalidad: patrones que se repiten cada 12 meses.')
print('    Si hay picos en ciertos meses (ej: diciembre), indica estacionalidad.')
print('  - Residuo: la variación no explicada por tendencia ni estacionalidad.')
print('    Si los residuos son grandes, hay shocks no predecibles.')

---

## Ejercicio 4: Prueba de estacionariedad (ADF)

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 4: Prueba ADF
# ============================================================

# 1. Aplicar la prueba de Dickey-Fuller Aumentada
#    H0: la serie tiene raíz unitaria (NO es estacionaria)
#    H1: la serie ES estacionaria
resultado_adf = adfuller(serie_inflacion)

# 2. Extraer e imprimir los resultados
estadistico_adf = resultado_adf[0]
p_value = resultado_adf[1]
valores_criticos = resultado_adf[4]

print('📊 Prueba de Dickey-Fuller Aumentada (ADF):')
print(f'  Estadístico ADF: {estadistico_adf:.4f}')
print(f'  P-value:         {p_value:.6f}')
print(f'\n  Valores críticos:')
for nivel, valor in valores_criticos.items():
    print(f'    {nivel}: {valor:.4f}')

# 3. Interpretar el resultado
print(f'\n📌 Interpretación:')
if p_value < 0.05:
    print(f'  P-value ({p_value:.6f}) < 0.05')
    print(f'  → Rechazamos H0: la serie ES estacionaria.')
    print(f'  → No necesita diferenciación para modelar con ARIMA.')
else:
    print(f'  P-value ({p_value:.6f}) >= 0.05')
    print(f'  → No podemos rechazar H0: la serie NO es estacionaria.')
    print(f'  → Necesita al menos una diferenciación (d=1) para ARIMA.')

# 4. Implicación para ARIMA
print(f'\n📌 Implicación para ARIMA:')
print(f'  - Si la serie es estacionaria, podemos usar d=0 en ARIMA(p,d,q).')
print(f'  - Si NO es estacionaria, usamos d=1 (o d=2) para diferenciar.')
print(f'  - La diferenciación hace estacionaria una serie con tendencia.')

---

## Ejercicio 5: Modelo ARIMA y pronóstico

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 5: Modelo ARIMA y pronóstico
# ============================================================

# 1. Seleccionar la serie sin NaN
serie = serie_inflacion.copy()
print(f'Serie para ARIMA: {len(serie)} observaciones')

# 2. Ajustar un modelo ARIMA(1,1,1)
#    p=1: un término autorregresivo (depende de su valor anterior)
#    d=1: una diferenciación (para hacer estacionaria la serie)
#    q=1: un término de media móvil (depende del error anterior)
modelo_arima = ARIMA(serie, order=(1, 1, 1))

# 3. Entrenar el modelo
resultado = modelo_arima.fit()

# Mostrar resumen del modelo
print('\n📊 Resumen del modelo ARIMA(1,1,1):')
print(f'  AIC: {resultado.aic:.2f}')
print(f'  BIC: {resultado.bic:.2f}')

# 4. Generar pronóstico de 6 meses
pronostico = resultado.forecast(steps=6)

# 5. Imprimir valores pronosticados
print(f'\n📊 Pronóstico de inflación para los próximos 6 meses:')
for fecha, valor in pronostico.items():
    print(f'  {fecha.strftime("%Y-%m")}: {valor:.2f}%')

print(f'\n📌 El modelo ARIMA(1,1,1) genera un pronóstico basado en')
print(f'   los patrones autorregresivos y de media móvil de la serie.')

---

## Ejercicio 6: Visualizar pronóstico con intervalos de confianza

In [ ]:
# ============================================================
# SOLUCIÓN Ejercicio 6: Visualizar pronóstico con intervalos
# ============================================================

# 1. Obtener pronóstico con intervalos de confianza
forecast_obj = resultado.get_forecast(steps=6)

# 2. Extraer valores predichos e intervalos de confianza
predichos = forecast_obj.predicted_mean
intervalo_confianza = forecast_obj.conf_int()

# 3. Crear el gráfico
fig, ax = plt.subplots(figsize=(14, 7))

# Serie original (últimos 24 meses para que se vea claro)
ultimos_24 = serie.tail(24)
ax.plot(ultimos_24.index, ultimos_24.values,
        color='steelblue', linewidth=2,
        label='Datos observados')

# Línea de pronóstico (en rojo)
ax.plot(predichos.index, predichos.values,
        color='red', linewidth=2, linestyle='--',
        marker='o', markersize=6,
        label='Pronóstico ARIMA(1,1,1)')

# Área de confianza al 95% (sombreada)
ax.fill_between(
    intervalo_confianza.index,
    intervalo_confianza.iloc[:, 0],  # límite inferior
    intervalo_confianza.iloc[:, 1],  # límite superior
    color='red', alpha=0.15,
    label='Intervalo de confianza 95%'
)

# 4. Título, leyenda y etiquetas
ax.set_xlabel('Fecha')
ax.set_ylabel('Inflación mensual (%)')
ax.set_title('Pronóstico de Inflación Mensual — ARIMA(1,1,1)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 5. Interpretación
ancho_intervalo = (intervalo_confianza.iloc[:, 1] - intervalo_confianza.iloc[:, 0]).mean()
print(f'📌 Interpretación del pronóstico:')
print(f'  - El modelo pronostica los valores de inflación para los próximos 6 meses.')
print(f'  - El ancho promedio del intervalo de confianza es {ancho_intervalo:.2f} puntos.')
if ancho_intervalo > 10:
    print(f'  - El intervalo es amplio, lo que indica alta incertidumbre.')
    print(f'    Esto es esperable en economías con alta volatilidad.')
else:
    print(f'  - El intervalo es relativamente estrecho, lo que sugiere')
    print(f'    confianza razonable en el pronóstico a corto plazo.')
print(f'  - A medida que el horizonte se extiende, el intervalo crece,')
print(f'    reflejando mayor incertidumbre en pronósticos más lejanos.')
print(f'  - En contextos de alta volatilidad como Venezuela, estos pronósticos')
print(f'    deben interpretarse con cautela y actualizarse frecuentemente.')

---

## 📋 Resumen

En este laboratorio practicaste:

| Paso | Qué hiciste |
|------|-------------|
| 1. Preparación | Cargaste y preparaste datos temporales con índice datetime |
| 2. Suavizamiento | Calculaste media móvil para revelar tendencia |
| 3. Descomposición | Separaste la serie en tendencia, estacionalidad y residuo |
| 4. Estacionariedad | Evaluaste con prueba ADF si la serie es estacionaria |
| 5. ARIMA | Ajustaste un modelo ARIMA para pronóstico |
| 6. Pronóstico | Visualizaste la predicción con intervalos de confianza |

## 📚 Referencias

1. Hyndman, R. J. & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts. https://otexts.com/fpp3/

2. Statsmodels developers. (2024). *Time Series Analysis*. https://www.statsmodels.org/stable/tsa.html

3. Box, G. E. P., Jenkins, G. M., Reinsel, G. C., & Ljung, G. M. (2015). *Time Series Analysis: Forecasting and Control* (5th ed.). Wiley.

4. Banco Central de Venezuela. *Indicadores Económicos*. https://www.bcv.org.ve/

---

*Notebook desarrollado para el programa de Formación Docente en Ciencia de Datos y Business Intelligence — FACES, Universidad Catolica Andres Bello (UCAB).*